features aus Mediapipe-Landmarks bauen 

---

In [1]:
# Imports
#  
import numpy as np
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from scipy.signal import resample
import sqlite3
import pandas as pd
import sqlite3
import pandas as pd
import duckdb
import polars as pl


In [2]:

# if the notebook is in the same folder as the db file:
con = duckdb.connect("landmark_database.db")

# otherwise:
# con = duckdb.connect(r"C:\Users\lejza\spice_bootcamp\landmark_database.db")


In [3]:
#  Load data as csv from MediaPipe Pose from database

#Connect to database
db_path =r"C:\Users\lejaz\spice_bootcamp\landmark_database.db"
conn = sqlite3.connect(db_path)

#==========================================
#BASIC QUERIES
#==========================================
#View all data (limit to first 1000 rows)
df = pd.read_sql_query("SELECT * FROM landmarks LIMIT 100000", conn)
print(f"Total rows loaded: {len(df)}")
df.head()

Test_data = df.copy()

#df.head()

#Rows: 21,850,000 - 22,050,000


Total rows loaded: 100000


In [4]:
#View all data (limit to first 1000 rows)
df = pd.read_sql_query("SELECT * FROM landmarks LIMIT 100000", conn)
print(f"Total rows loaded: {len(df)}")
df.head()

Total rows loaded: 100000


,id,patient_name,frame,movement_type,jacket_status,side,model_name,timestamp_ms,landmark_id,x_norm,...,title,uploader,fps,start_time,end_time,duration,checksum,width,height,created_at
0,1,PA000,0,Fast Movement,With Jacket,Right,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38
1,2,PA000,1,Fast Movement,With Jacket,Right,DensePose,33.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38
2,3,PA000,2,Fast Movement,With Jacket,Right,DensePose,66.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38
3,4,PA000,3,Fast Movement,With Jacket,Right,DensePose,100.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38
4,5,PA000,4,Fast Movement,With Jacket,Right,DensePose,133.0,0,NaN,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:38


In [5]:
Test_data.tail(30)

,id,patient_name,frame,movement_type,jacket_status,side,model_name,timestamp_ms,landmark_id,x_norm,...,title,uploader,fps,start_time,end_time,duration,checksum,width,height,created_at
99970,99971,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,21,0.711586,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99971,99972,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,22,0.725127,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99972,99973,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,23,0.696310,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99973,99974,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,24,0.701167,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99974,99975,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,25,0.707770,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99975,99976,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,26,0.704635,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99976,99977,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,27,0.666555,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99977,99978,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,28,0.692548,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99978,99979,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,29,0.651192,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41
99979,99980,PA001,231,Regular Movement,Without Jacket,Left,DensePose,7700.0,30,0.683132,...,None,None,None,None,None,None,None,None,None,2026-01-14 16:47:41


---

keep gait_preprocessing_pipeline.py as authoritative module

In [6]:
import gait_preprocessing_pipeline as gait


- turn the long-format MediaPipe data into a df_video
    - one row per video with a pose tensor

- take one normalized gait clip, e.g. (T,33,3), and return a dict of scalar gait features

- df_video als preprocess into clips via the given pipelien and than with that a feature dataframe

In [7]:
%load_ext autoreload
%autoreload 2

import feature_extraction as fx
import gait_preprocessing_pipeline as gait
import numpy as np
import pandas as pd


In [8]:
%reload_ext autoreload
%autoreload 2
import importlib
importlib.reload(fx)


<module 'feature_extraction' from 'c:\\Users\\lejaz\\spice_bootcamp\\GAITy-Capstone-Modeling\\notebooks\\feature_extraction.py'>

In [10]:
dummy_clip = np.zeros((60, gait.N_JOINTS, 3), dtype=np.float32)
fx.compute_clip_features(dummy_clip)

TypeError: compute_clip_features() missing 1 required positional argument: 'fps'

In [11]:
whos


Variable            Type                  Data/Info
---------------------------------------------------
Test_data           DataFrame             Shape: (100000, 35)
con                 DuckDBPyConnection    <_duckdb.DuckDBPyConnecti<...>ct at 0x00000204031AA970>
conn                Connection            <sqlite3.Connection object at 0x0000020463EF3B50>
db_path             str                   C:\Users\lejaz\spice_boot<...>camp\landmark_database.db
df                  DataFrame             Shape: (100000, 35)
duckdb              module                <module 'duckdb' from 'c:<...>es\\duckdb\\__init__.py'>
dummy_clip          ndarray               60x33x3: 5940 elems, type `float32`, 23760 bytes
find_peaks          function              <function find_peaks at 0x00000204025A62A0>
fx                  module                <module 'feature_extracti<...>\\feature_extraction.py'>
gait                module                <module 'gait_preprocessi<...>eprocessing_pipeline.py'>
gaussian_filt

In [12]:
from gait_preprocessing_pipeline import add_pose_column

df_video = add_pose_column(Test_data)
df_video.head()


,id,patient_name,frame,movement_type,jacket_status,side,model_name,timestamp_ms,landmark_id,x_norm,...,uploader,fps,start_time,end_time,duration,checksum,width,height,created_at,pose
0,1,PA000,0,Fast Movement,With Jacket,Right,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:38,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."
5504,5505,PA000,0,Fast Movement,With Jacket,Left,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:38,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."
10902,10903,PA000,0,Fast Movement,Without Jacket,Right,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:39,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."
15616,15617,PA000,0,Fast Movement,Without Jacket,Left,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:39,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."
21008,21009,PA000,0,Regular Movement,With Jacket,Right,DensePose,0.0,0,NaN,...,None,None,None,None,None,None,None,None,2026-01-14 16:47:39,"[[[nan, nan, nan], [0.0, 0.0, 0.0], [0.0, 0.0,..."


In [13]:
df_video.iloc[0]["pose"].shape


(192, 33, 3)

video level dataframe is ready

---

In [14]:
df_video.columns


Index(['id', 'patient_name', 'frame', 'movement_type', 'jacket_status', 'side',
       'model_name', 'timestamp_ms', 'landmark_id', 'x_norm', 'y_norm',
       'z_norm', 'visibility', 'x_px', 'y_px', 'source_file', 'file_order',
       'file_path', 'start_frame', 'end_frame', 'url', 'gait_event', 'dataset',
       'gait_pattern', 'add_pattern_info', 'title', 'uploader', 'fps',
       'start_time', 'end_time', 'duration', 'checksum', 'width', 'height',
       'created_at', 'pose'],
      dtype='object')

In [15]:
# 1) check pose shape
df_video.iloc[0]["pose"].shape

# 2) check what dataset labels look like
df_video["dataset"].value_counts()

# 3) rough fps overview
df_video["fps"].describe()


count       0
unique      0
top       NaN
freq      NaN
Name: fps, dtype: object

In [16]:
label_map_for_strings = {
    "normal": "normal gait",
    "abnormal": "abnormal gait",
    "healthy": "normal gait",
    "pathological": "abnormal gait",
    # add whatever you actually have
}

df_video["dataset"] = df_video["dataset"].map(label_map_for_strings)




from xgboost import XGBClassifier

X = df_features.drop(columns=["label"])
y = df_features["label"]

model = XGBClassifier()
model.fit(X, y)


---

now the actual feature calculation! :) 

---

In [17]:
df_video["movement_type"].value_counts()


movement_type
Fast Movement       8
Regular Movement    8
Name: count, dtype: int64

In [18]:
df_video["side"].value_counts()


side
Right    8
Left     8
Name: count, dtype: int64

In [19]:
df_video["gait_event"].value_counts()


Series([], Name: count, dtype: int64)

---

## set up generic base functions for any landmarker (set) to calculate the functions
joint_speed => underlying speed signal
moving_and_still times => how long moves/not moves
range_of_motion => ROM for a joint

In [20]:
%load_ext autoreload
%autoreload 2

import feature_extraction as fx
import gait_preprocessing_pipeline as gait
import numpy as np

dummy_clip = np.zeros((60, gait.N_JOINTS, 3), dtype=np.float32)
fx.compute_clip_features(dummy_clip, fps=30.0)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


{'step_height_L': 0.0,
 'step_height_R': 0.0,
 'step_length_L': 0.0,
 'step_length_R': 0.0,
 'pelvis_drop_mean': 0.0,
 'pelvis_drop_std': 0.0,
 'trunk_lean_mean': 0.0,
 'trunk_lean_std': 0.0,
 'heel_range_L': 0.0,
 'heel_range_R': 0.0,
 'step_height_symmetry': 0.0,
 'step_length_symmetry': 0.0,
 'knee_L_moving_time_sec': 0.0,
 'knee_L_still_time_sec': 1.9666666666666666,
 'knee_L_moving_fraction': 0.0,
 'knee_L_still_fraction': 1.0,
 'knee_L_mean_speed': 0.0,
 'knee_L_max_speed': 0.0,
 'knee_L_total_time_sec': 1.9666666666666666,
 'knee_R_moving_time_sec': 0.0,
 'knee_R_still_time_sec': 1.9666666666666666,
 'knee_R_moving_fraction': 0.0,
 'knee_R_still_fraction': 1.0,
 'knee_R_mean_speed': 0.0,
 'knee_R_max_speed': 0.0,
 'knee_R_total_time_sec': 1.9666666666666666,
 'knee_L_rom_y': 0.0,
 'knee_R_rom_y': 0.0,
 'hip_L_rom_y': 0.0,
 'hip_R_rom_y': 0.0,
 'shoulder_L_rom_x': 0.0,
 'shoulder_R_rom_x': 0.0,
 'ankle_L_rom_y': 0.0,
 'ankle_R_rom_y': 0.0,
 'knee_rom_asym': 0.0,
 'hip_rom_asym': 

In [21]:
[k for k in fx.compute_clip_features(dummy_clip, 30.0).keys() if "knee" in k]


['knee_L_moving_time_sec',
 'knee_L_still_time_sec',
 'knee_L_moving_fraction',
 'knee_L_still_fraction',
 'knee_L_mean_speed',
 'knee_L_max_speed',
 'knee_L_total_time_sec',
 'knee_R_moving_time_sec',
 'knee_R_still_time_sec',
 'knee_R_moving_fraction',
 'knee_R_still_fraction',
 'knee_R_mean_speed',
 'knee_R_max_speed',
 'knee_R_total_time_sec',
 'knee_L_rom_y',
 'knee_R_rom_y',
 'knee_rom_asym']

In [22]:
['knee_L_moving_time_sec', 'knee_L_still_time_sec', ..., 'knee_L_rom_y',
 'knee_R_moving_time_sec', ..., 'knee_R_rom_y']


['knee_L_moving_time_sec',
 'knee_L_still_time_sec',
 Ellipsis,
 'knee_L_rom_y',
 'knee_R_moving_time_sec',
 Ellipsis,
 'knee_R_rom_y']

In [23]:
df_features = fx.extract_features_from_df_video(df_video)
df_features.head()


,step_height_L,step_height_R,step_length_L,step_length_R,pelvis_drop_mean,pelvis_drop_std,trunk_lean_mean,trunk_lean_std,heel_range_L,heel_range_R,...,ankle_R_still_fraction,stance_ratio_L,stance_ratio_R,stance_ratio_asym,label_fine,label_class,label_id,movement_type,side,source_file
0,1.726805,1.695221,1.346603,1.138875,0.020217,0.024436,-0.030305,0.147717,1.919934,1.857972,...,0.115183,0.130177,0.130177,0.0,None,None,None,Fast Movement,Right,semantic_segmentation_PA000_FGS_WJ_1_DensePose...
1,2.526946,3.045800,1.524194,1.609756,0.015201,0.083514,-0.054901,0.113512,2.644770,3.116001,...,0.055249,0.058479,0.058479,0.0,None,None,None,Fast Movement,Left,semantic_segmentation_PA000_FGS_WJ_2_DensePose...
2,1.819934,1.783279,1.225501,1.201241,0.011532,0.021964,0.008385,0.158382,1.991898,1.952520,...,0.130178,0.149660,0.149660,0.0,None,None,None,Fast Movement,Right,semantic_segmentation_PA000_FGS_WoJ_1_DensePos...
3,2.067794,2.718333,1.551333,1.546484,0.001384,0.066918,-0.039065,0.135969,2.124789,2.972578,...,0.005714,0.005747,0.005747,0.0,None,None,None,Fast Movement,Left,semantic_segmentation_PA000_FGS_WoJ_2_DensePos...
4,1.810680,1.735085,1.193693,1.035499,0.018124,0.019447,-0.041142,0.152405,1.958019,1.903667,...,0.129353,0.148571,0.148571,0.0,None,None,None,Regular Movement,Right,semantic_segmentation_PA000_UGS_WJ_1_DensePose...


In [24]:
df_features.head()
df_features.columns


Index(['step_height_L', 'step_height_R', 'step_length_L', 'step_length_R',
       'pelvis_drop_mean', 'pelvis_drop_std', 'trunk_lean_mean',
       'trunk_lean_std', 'heel_range_L', 'heel_range_R',
       'step_height_symmetry', 'step_length_symmetry',
       'knee_L_moving_time_sec', 'knee_L_still_time_sec',
       'knee_L_moving_fraction', 'knee_L_still_fraction', 'knee_L_mean_speed',
       'knee_L_max_speed', 'knee_L_total_time_sec', 'knee_R_moving_time_sec',
       'knee_R_still_time_sec', 'knee_R_moving_fraction',
       'knee_R_still_fraction', 'knee_R_mean_speed', 'knee_R_max_speed',
       'knee_R_total_time_sec', 'knee_L_rom_y', 'knee_R_rom_y', 'hip_L_rom_y',
       'hip_R_rom_y', 'shoulder_L_rom_x', 'shoulder_R_rom_x', 'ankle_L_rom_y',
       'ankle_R_rom_y', 'knee_rom_asym', 'hip_rom_asym', 'shoulder_rom_asym',
       'ankle_rom_asym', 'ankle_L_moving_fraction', 'ankle_L_still_fraction',
       'ankle_R_moving_fraction', 'ankle_R_still_fraction', 'stance_ratio_L',
       'st

- this results in completely numerical feature columns
- labels: label_fine, label_class, label_id
- meta: movement_type, side, source_file 

Based on this, a multi class basmodel can be build. 

---


In [27]:
%load_ext autoreload
%autoreload 2

import feature_extraction as fx
import gait_preprocessing_pipeline as gait
import numpy as np

dummy_clip = np.zeros((60, gait.N_JOINTS, 3), dtype=np.float32)
feats = fx.compute_clip_features(dummy_clip, fps=30.0)
[k for k in feats.keys() if "angle" in k]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


['knee_angle_L_mean',
 'knee_angle_L_std',
 'knee_angle_L_rom',
 'knee_angle_R_mean',
 'knee_angle_R_std',
 'knee_angle_R_rom',
 'hip_angle_L_mean',
 'hip_angle_L_std',
 'hip_angle_L_rom',
 'hip_angle_R_mean',
 'hip_angle_R_std',
 'hip_angle_R_rom',
 'ankle_angle_L_mean',
 'ankle_angle_L_std',
 'ankle_angle_L_rom',
 'ankle_angle_R_mean',
 'ankle_angle_R_std',
 'ankle_angle_R_rom',
 'knee_angle_rom_asym',
 'hip_angle_rom_asym',
 'ankle_angle_rom_asym']

In [28]:
df_features = fx.extract_features_from_df_video(df_video)
df_features.columns


Index(['step_height_L', 'step_height_R', 'step_length_L', 'step_length_R',
       'pelvis_drop_mean', 'pelvis_drop_std', 'trunk_lean_mean',
       'trunk_lean_std', 'heel_range_L', 'heel_range_R',
       'step_height_symmetry', 'step_length_symmetry',
       'knee_L_moving_time_sec', 'knee_L_still_time_sec',
       'knee_L_moving_fraction', 'knee_L_still_fraction', 'knee_L_mean_speed',
       'knee_L_max_speed', 'knee_L_total_time_sec', 'knee_R_moving_time_sec',
       'knee_R_still_time_sec', 'knee_R_moving_fraction',
       'knee_R_still_fraction', 'knee_R_mean_speed', 'knee_R_max_speed',
       'knee_R_total_time_sec', 'knee_L_rom_y', 'knee_R_rom_y', 'hip_L_rom_y',
       'hip_R_rom_y', 'shoulder_L_rom_x', 'shoulder_R_rom_x', 'ankle_L_rom_y',
       'ankle_R_rom_y', 'knee_rom_asym', 'hip_rom_asym', 'shoulder_rom_asym',
       'ankle_rom_asym', 'ankle_L_moving_fraction', 'ankle_L_still_fraction',
       'ankle_R_moving_fraction', 'ankle_R_still_fraction', 'stance_ratio_L',
       'st

In [29]:
%load_ext autoreload
%autoreload 2

import feature_extraction as fx
import gait_preprocessing_pipeline as gait
import numpy as np

dummy_clip = np.zeros((120, gait.N_JOINTS, 3), dtype=np.float32)
feats = fx.compute_clip_features(dummy_clip, fps=30.0)
[k for k in feats.keys() if "step_" in k or "cadence" in k or "width" in k]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


['step_height_L',
 'step_height_R',
 'step_length_L',
 'step_length_R',
 'step_height_symmetry',
 'step_length_symmetry',
 'step_L_mean_step_time',
 'step_L_std_step_time',
 'step_L_cadence',
 'step_L_mean_stride_time',
 'step_L_std_stride_time',
 'step_L_step_time_cv',
 'step_R_mean_step_time',
 'step_R_std_step_time',
 'step_R_cadence',
 'step_R_mean_stride_time',
 'step_R_std_stride_time',
 'step_R_step_time_cv',
 'step_time_asym',
 'cadence_asym',
 'step_width_mean',
 'step_width_std']

In [30]:
df_features = fx.extract_features_from_df_video(df_video)
df_features.filter(regex="step_|cadence|width|stance_ratio").head()


,step_height_L,step_height_R,step_length_L,step_length_R,step_height_symmetry,step_length_symmetry,stance_ratio_L,stance_ratio_R,stance_ratio_asym,step_L_mean_step_time,...,step_R_mean_step_time,step_R_std_step_time,step_R_cadence,step_R_mean_stride_time,step_R_std_stride_time,step_R_step_time_cv,step_time_asym,cadence_asym,step_width_mean,step_width_std
0,1.726805,1.695221,1.346603,1.138875,0.009229,0.083577,0.130177,0.130177,0.0,0.441667,...,0.484848,0.139526,123.750000,0.980000,0.191021,0.287772,-0.046607,0.046607,0.462231,0.348903
1,2.526946,3.045800,1.524194,1.609756,-0.093106,-0.027302,0.058479,0.058479,0.0,0.533333,...,0.427778,0.089062,140.259740,0.869697,0.107736,0.208198,0.109826,-0.109827,0.437087,0.331132
2,1.819934,1.783279,1.225501,1.201241,0.010173,0.009997,0.149660,0.149660,0.0,0.470370,...,0.474074,0.177623,126.562500,0.987500,0.199261,0.374674,-0.003922,0.003922,0.471943,0.351158
3,2.067794,2.718333,1.551333,1.546484,-0.135922,0.001565,0.005747,0.005747,0.0,0.566667,...,0.503333,0.129486,119.205298,0.992593,0.184443,0.257257,0.059190,-0.059190,0.451288,0.327425
4,1.810680,1.735085,1.193693,1.035499,0.021320,0.070965,0.148571,0.148571,0.0,0.460606,...,0.473333,0.097525,126.760563,0.933333,0.157919,0.206039,-0.013628,0.013628,0.452362,0.354423


---

overview of all features 

In [31]:
df_features.columns.tolist()


['step_height_L',
 'step_height_R',
 'step_length_L',
 'step_length_R',
 'pelvis_drop_mean',
 'pelvis_drop_std',
 'trunk_lean_mean',
 'trunk_lean_std',
 'heel_range_L',
 'heel_range_R',
 'step_height_symmetry',
 'step_length_symmetry',
 'knee_L_moving_time_sec',
 'knee_L_still_time_sec',
 'knee_L_moving_fraction',
 'knee_L_still_fraction',
 'knee_L_mean_speed',
 'knee_L_max_speed',
 'knee_L_total_time_sec',
 'knee_R_moving_time_sec',
 'knee_R_still_time_sec',
 'knee_R_moving_fraction',
 'knee_R_still_fraction',
 'knee_R_mean_speed',
 'knee_R_max_speed',
 'knee_R_total_time_sec',
 'knee_L_rom_y',
 'knee_R_rom_y',
 'hip_L_rom_y',
 'hip_R_rom_y',
 'shoulder_L_rom_x',
 'shoulder_R_rom_x',
 'ankle_L_rom_y',
 'ankle_R_rom_y',
 'knee_rom_asym',
 'hip_rom_asym',
 'shoulder_rom_asym',
 'ankle_rom_asym',
 'ankle_L_moving_fraction',
 'ankle_L_still_fraction',
 'ankle_R_moving_fraction',
 'ankle_R_still_fraction',
 'stance_ratio_L',
 'stance_ratio_R',
 'stance_ratio_asym',
 'knee_angle_L_mean',
 '

In [32]:
for c in df_features.columns:
    print(c)


step_height_L
step_height_R
step_length_L
step_length_R
pelvis_drop_mean
pelvis_drop_std
trunk_lean_mean
trunk_lean_std
heel_range_L
heel_range_R
step_height_symmetry
step_length_symmetry
knee_L_moving_time_sec
knee_L_still_time_sec
knee_L_moving_fraction
knee_L_still_fraction
knee_L_mean_speed
knee_L_max_speed
knee_L_total_time_sec
knee_R_moving_time_sec
knee_R_still_time_sec
knee_R_moving_fraction
knee_R_still_fraction
knee_R_mean_speed
knee_R_max_speed
knee_R_total_time_sec
knee_L_rom_y
knee_R_rom_y
hip_L_rom_y
hip_R_rom_y
shoulder_L_rom_x
shoulder_R_rom_x
ankle_L_rom_y
ankle_R_rom_y
knee_rom_asym
hip_rom_asym
shoulder_rom_asym
ankle_rom_asym
ankle_L_moving_fraction
ankle_L_still_fraction
ankle_R_moving_fraction
ankle_R_still_fraction
stance_ratio_L
stance_ratio_R
stance_ratio_asym
knee_angle_L_mean
knee_angle_L_std
knee_angle_L_rom
knee_angle_R_mean
knee_angle_R_std
knee_angle_R_rom
hip_angle_L_mean
hip_angle_L_std
hip_angle_L_rom
hip_angle_R_mean
hip_angle_R_std
hip_angle_R_rom
an

In [34]:
df_features.filter(regex="rom").columns
df_features.filter(regex="angle").columns
df_features.filter(regex="step_").columns
df_features.filter(regex="cadence").columns
df_features.filter(regex="width").columns
df_features.filter(regex="stance").columns
df_features.filter(regex="asym").columns


Index(['knee_rom_asym', 'hip_rom_asym', 'shoulder_rom_asym', 'ankle_rom_asym',
       'stance_ratio_asym', 'knee_angle_rom_asym', 'hip_angle_rom_asym',
       'ankle_angle_rom_asym', 'step_time_asym', 'cadence_asym'],
      dtype='object')

In [35]:
len(feature_cols)


82

In [38]:
df_features[feature_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
step_height_L,16.0,2.057925,0.525174,1.582300,1.740909,1.857411,2.154066,3.678692
step_height_R,16.0,2.164843,0.592260,1.593982,1.725119,1.882352,2.643532,3.428821
step_length_L,16.0,1.201082,0.224970,0.781213,1.101783,1.234889,1.351279,1.551333
step_length_R,16.0,1.156485,0.212871,0.892823,0.993149,1.143836,1.228654,1.609756
pelvis_drop_mean,16.0,0.007783,0.013115,-0.012641,-0.005082,0.012784,0.020285,0.023695
...,...,...,...,...,...,...,...,...
step_R_step_time_cv,16.0,0.300600,0.071465,0.189517,0.254858,0.299165,0.345281,0.423717
step_time_asym,16.0,0.003394,0.071552,-0.111517,-0.047244,-0.008775,0.062223,0.138046
cadence_asym,16.0,-0.003394,0.071552,-0.138047,-0.062224,0.008775,0.047244,0.111517
step_width_mean,16.0,0.419308,0.053289,0.317010,0.405584,0.441259,0.451557,0.471943


In [39]:
df_features.columns.tolist()



['step_height_L',
 'step_height_R',
 'step_length_L',
 'step_length_R',
 'pelvis_drop_mean',
 'pelvis_drop_std',
 'trunk_lean_mean',
 'trunk_lean_std',
 'heel_range_L',
 'heel_range_R',
 'step_height_symmetry',
 'step_length_symmetry',
 'knee_L_moving_time_sec',
 'knee_L_still_time_sec',
 'knee_L_moving_fraction',
 'knee_L_still_fraction',
 'knee_L_mean_speed',
 'knee_L_max_speed',
 'knee_L_total_time_sec',
 'knee_R_moving_time_sec',
 'knee_R_still_time_sec',
 'knee_R_moving_fraction',
 'knee_R_still_fraction',
 'knee_R_mean_speed',
 'knee_R_max_speed',
 'knee_R_total_time_sec',
 'knee_L_rom_y',
 'knee_R_rom_y',
 'hip_L_rom_y',
 'hip_R_rom_y',
 'shoulder_L_rom_x',
 'shoulder_R_rom_x',
 'ankle_L_rom_y',
 'ankle_R_rom_y',
 'knee_rom_asym',
 'hip_rom_asym',
 'shoulder_rom_asym',
 'ankle_rom_asym',
 'ankle_L_moving_fraction',
 'ankle_L_still_fraction',
 'ankle_R_moving_fraction',
 'ankle_R_still_fraction',
 'stance_ratio_L',
 'stance_ratio_R',
 'stance_ratio_asym',
 'knee_angle_L_mean',
 '

In [40]:
len(df_features.columns)


88

---

In [43]:
# Wie viele NaNs pro Spalte?
df_features.isna().sum().sort_values(ascending=False).head(20)



label_id              16
label_class           16
label_fine            16
step_height_L          0
hip_angle_R_mean       0
ankle_angle_R_rom      0
ankle_angle_R_std      0
ankle_angle_R_mean     0
ankle_angle_L_rom      0
ankle_angle_L_std      0
ankle_angle_L_mean     0
hip_angle_R_rom        0
hip_angle_R_std        0
hip_angle_L_std        0
hip_angle_L_rom        0
hip_angle_rom_asym     0
hip_angle_L_mean       0
knee_angle_R_rom       0
knee_angle_R_std       0
knee_angle_R_mean      0
dtype: int64

In [44]:
nunique = df_features.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
constant_cols


['stance_ratio_asym', 'label_fine', 'label_class', 'label_id']